In [ ]:
# Option B: construct model manually (paste into Cell 2)
import sys
sys.path.append("/mnt/d/Dev/FSS/DCAMA_project/")  # se necessário

# exemplo 1: import class e construir
from DCAMA.model.DCAMA_new import DCAMA as MyModel  # ajuste o import conforme seu código
model = MyModel(backbone='swin', pretrained_path="../backbones/swin.pth", use_original_imgsize=False)

# # exemplo 2: criar uma rede simples para teste (se quiser apenas testar o counting)
# import torch.nn as nn
# model = nn.Sequential(
#     nn.Conv2d(3,64,3,padding=1),
#     nn.ReLU(),
#     nn.Conv2d(64,64,3,padding=1),
#     nn.AdaptiveAvgPool2d((1,1)),
#     nn.Flatten(),
#     nn.Linear(64,10)
# )

model_cpu = model.to("cpu")
print("Model constructed. Top-level children:", [n for n,_ in model_cpu.named_children()])

In [40]:
model

DCAMA(
  (feature_extractor): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0): BasicLayer(
        dim=128, input_resolution=(96, 96), depth=2
        (blocks): ModuleList(
          (0): SwinTransformerBlock(
            dim=128, input_resolution=(96, 96), num_heads=4, window_size=12, shift_size=0, mlp_ratio=4.0
            (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              dim=128, window_size=(12, 12), num_heads=4
              (qkv): Linear(in_features=128, out_features=384, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=128, out_features=128, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(

In [41]:
# Cell 1: Imports, config and helper functions
import importlib
import os
import torch
import inspect
import pandas as pd
import numpy as np
import csv
import time
from collections import OrderedDict

# ---------- Configuration (edit these before running) ----------
# Option A: Importable builder function in the form "module.path:callable"
# Example: "myrepo.models:build_model"  (the callable should return an nn.Module)
MODEL_MODULE = model  # e.g. "myrepo.models:build_model"
# Option B: If you want to construct your model manually in the cell below, set MODEL_MODULE = None
# and provide model construction code in the "Build or import model" cell.

# Optional checkpoint path (set to None to skip loading)
ckp_pth = "/mnt/d/Dev/FSS/DCAMA_project/DCAMA/logs/train/Swin_1s_Pascal_SE/best_model.pt"
CHECKPOINT_PATH = ckp_pth  # e.g. "/path/to/checkpoint.pth"

# Device used for any optional timing / FLOPs measurements
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Substrings to search param names for (useful to find backbone, u2net etc.)
SUBSTRING_KEYS = ["backbone","resnet","u2","u2net","u2_net","saliency","decoder",
                  "mask","attn","attention","cross","mixer","head"]

# Where to write CSV for LaTeX conversion
OUTPUT_CSV = "params_table_input.csv"

# ---------- Helper functions ----------
def count_params(model: torch.nn.Module):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return int(total), int(trainable)

def params_by_top_children(model: torch.nn.Module):
    """
    Returns an OrderedDict mapping top-level child name -> (total_params, trainable_params)
    Uses model.named_children() (only top-level)
    """
    results = OrderedDict()
    for name, module in model.named_children():
        tot = sum(p.numel() for p in module.parameters())
        tr = sum(p.numel() for p in module.parameters() if p.requires_grad)
        results[name] = (int(tot), int(tr))
    return results

def params_by_substring(model: torch.nn.Module, substrings=SUBSTRING_KEYS):
    """
    Search named_parameters() for substrings and aggregate.
    Returns dict substring -> (total, trainable)
    """
    out = {}
    for s in substrings:
        tot = 0
        tr = 0
        for name,p in model.named_parameters():
            if s in name.lower():
                tot += p.numel()
                if p.requires_grad:
                    tr += p.numel()
        out[s] = (int(tot), int(tr))
    return out

def try_load_checkpoint(model, ckpt_path):
    if ckpt_path is None or not os.path.exists(ckpt_path):
        print("No checkpoint loaded (path is None or does not exist).")
        return model, None
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = None
    if isinstance(ckpt, dict):
        # try common keys
        for key in ("model","state_dict","state_dict_ema","net"):
            if key in ckpt:
                sd = ckpt[key]
                break
        if sd is None:
            sd = ckpt
    else:
        sd = ckpt
    # deal with module. prefix
    new_sd = {}
    if isinstance(sd, dict):
        for k,v in sd.items():
            new_k = k
            if k.startswith("module."):
                new_k = k[len("module."):]
            new_sd[new_k] = v
    else:
        new_sd = sd
    try:
        model.load_state_dict(new_sd, strict=False)
        print("Checkpoint loaded into model (strict=False).")
    except Exception as e:
        print("Warning: load_state_dict failed or partial. Exception:", e)
    return model, new_sd


In [42]:
# Optional checkpoint path (set to None to skip loading)
ckp_pth = "/mnt/d/Dev/FSS/DCAMA_project/DCAMA/logs/train/Swin_1s_SE_new/best_model.pt"
CHECKPOINT_PATH = ckp_pth  # e.g. "/path/to/checkpoint.pth"

model_cpu, loaded_sd = try_load_checkpoint(model_cpu, CHECKPOINT_PATH)

Checkpoint loaded into model (strict=False).


In [43]:
# Cell 3: optional load checkpoint, count params, and show breakdown
model_cpu, loaded_sd = try_load_checkpoint(model_cpu, CHECKPOINT_PATH)

total, trainable = count_params(model_cpu)
print(f"Total params: {total:,}")
print(f"Trainable params: {trainable:,}\n")

# Top-level children
child_counts = params_by_top_children(model_cpu)
df_children = pd.DataFrame([
    {"Module": name, "TotalParams": vals[0], "TrainableParams": vals[1]}
    for name, vals in child_counts.items()
])
display(df_children.style.format({"TotalParams":"{:,}", "TrainableParams":"{:,}"}))

# Substring breakdown
substr_break = params_by_substring(model_cpu)
df_sub = pd.DataFrame([
    {"Substring": k, "TotalParams": v[0], "TrainableParams": v[1]}
    for k,v in substr_break.items()
])
display(df_sub.style.format({"TotalParams":"{:,}", "TrainableParams":"{:,}"}))


Checkpoint loaded into model (strict=False).
Total params: 95,076,178
Trainable params: 95,076,178



,Module,TotalParams,TrainableParams
0,feature_extractor,"87,903,584","87,903,584"
1,model,"7,172,594","7,172,594"
2,cross_entropy_loss,0,0


,Substring,TotalParams,TrainableParams
0,backbone,0,0
1,resnet,0,0
2,u2,0,0
3,u2net,0,0
4,u2_net,0,0
5,saliency,0,0
6,decoder,0,0
7,mask,0,0
8,attn,"28,165,368","28,165,368"
9,attention,0,0


In [44]:
# Cell 4: Export CSV structured for csv_to_latex_table.py
# This makes one CSV row per top-level child (you can refine later)
rows = []
for name, (tot, tr) in child_counts.items():
    # friendly block mapping suggestion - update mapping if you have known submodule names
    friendly = name
    params_field = f"{tot:,} | {tr:,}"
    rows.append({
        "Block": friendly if friendly.strip() else "-",
        "Layer": name,
        "Input Size": "",
        "Output Size": "",
        "Params": params_field,
        "Activation": ""
    })

# add overall total row
rows.append({
    "Block": "-", "Layer": "Total", "Input Size": "", "Output Size": "",
    "Params": f"{total:,} | {trainable:,}", "Activation": ""
})

# write CSV
fieldnames = ["Block","Layer","Input Size","Output Size","Params","Activation"]
with open(OUTPUT_CSV,"w",newline="",encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in rows:
        writer.writerow(r)
print(f"Wrote CSV for LaTeX converter: {OUTPUT_CSV}")

# show the CSV as dataframe
display(pd.DataFrame(rows).style.format({"Params":str}))


Wrote CSV for LaTeX converter: params_table_input.csv


,Block,Layer,Input Size,Output Size,Params,Activation
0,feature_extractor,feature_extractor,,,"87,903,584 | 87,903,584",
1,model,model,,,"7,172,594 | 7,172,594",
2,cross_entropy_loss,cross_entropy_loss,,,0 | 0,
3,-,Total,,,"95,076,178 | 95,076,178",


In [46]:
# Verificações iniciais: top-level children e contagem total
print("Top-level children:", [n for n,_ in model.named_children()])
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,}, Trainable params: {trainable:,}")


Top-level children: ['feature_extractor', 'model', 'cross_entropy_loss']
Total params: 95,076,178, Trainable params: 95,076,178


In [49]:
from collections import defaultdict

def group_by_prefix(rows, depth=1):
    # depth: quantas partes do nome usar como "módulo" (separa por '.')
    d = defaultdict(lambda: [0,0])  # total, trainable
    for name, shape, numel, req in rows:
        key = ".".join(name.split(".")[:depth])
        d[key][0] += numel
        d[key][1] += (numel if req else 0)
    return d

# experimente depths 1..5 para achar agrupamento que faça sentido para seu modelo
for depth in range(1,5):
    print("Depth", depth)
    d = group_by_prefix(rows, depth)
    # mostrar top 20 módulos por size
    items = sorted(d.items(), key=lambda x: x[1][0], reverse=True)[:20]
    for k,(tot,tr) in items:
        print(f"{k:40s} : total {tot:,} | trainable {tr:,}")
    print()


Depth 1
feature_extractor                        : total 87,903,584 | trainable 87,903,584
model                                    : total 7,172,594 | trainable 7,172,594

Depth 2
feature_extractor.layers                 : total 86,870,008 | trainable 86,870,008
model.DCAMA_blocks                       : total 2,756,096 | trainable 2,756,096
model.DCAMA_multhead                     : total 2,099,200 | trainable 2,099,200
model.mixer1                             : total 1,106,112 | trainable 1,106,112
feature_extractor.head                   : total 1,025,000 | trainable 1,025,000
model.conv4                              : total 443,520 | trainable 443,520
model.conv5                              : total 443,520 | trainable 443,520
model.conv3                              : total 100,752 | trainable 100,752
model.conv2                              : total 90,768 | trainable 90,768
model.conv1                              : total 83,856 | trainable 83,856
model.mixer2                   

In [ ]:
import csv
depth = 4   # ajuste para depth que melhor agrupa seus módulos
d = group_by_prefix(rows, depth)
with open('params_by_module_depth'+str(depth)+'.csv','w',newline='',encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["ModulePrefix","TotalParams","TrainableParams"])
    for k,(tot,tr) in sorted(d.items(), key=lambda x: x[1][0], reverse=True):
        writer.writerow([k, tot, tr])
print(f"Wrote params_by_module_depth{depth}.csv")

Wrote params_by_module_depth5.csv


In [56]:
import csv
with open('params_by_parameter.csv','w',newline='',encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["ParamName","Shape","Numel","Trainable"])
    for name, shape, numel, req in rows:
        writer.writerow([name, str(shape), numel, int(req)])
print("Wrote params_by_parameter.csv")


Wrote params_by_parameter.csv


## -------------------------------------------------

In [67]:
# ============================
# Script: params_to_csv_with_depth.py (Jupyter-ready)
# ============================
# Objetivo: gerar:
#  1) params_by_parameter.csv  -> linha por parâmetro (name, shape, numel, trainable)
#  2) params_by_module_depth{depth}.csv -> linhas agregadas por prefixo (depth configurável)
# Uso: colar em Jupyter após ter o objeto `model` em memória.
# Ajuste as configurações abaixo conforme necessário.
# ============================

import csv
import os
from collections import defaultdict, OrderedDict

# ====== CONFIGURAÇÃO - edite aqui ======
# Depth: número de segmentos do nome separados por '.' usados para agrupar (ex: 1,2,3,4)
DEPTH = 4

# Top K módulos para incluir no CSV agrupado (use None para incluir todos)
TOP_K = None  # ex: 20 or None

# Se quiser congelar (freeze) módulos antes de contar, informe prefixos aqui
PREFIXES_TO_FREEZE = []  # ex: ['feature_extractor', 'backbone', 'u2net']

# Remove prefixo 'module.' típico de DataParallel
CLEAN_MODULE_PREFIX = True

# Nomes dos arquivos de saída
OUT_CSV_PER_PARAM = "params_by_parameter.csv"
OUT_CSV_BY_MODULE = f"params_by_module_depth{DEPTH}.csv"

# =========================================

# Verifica se model existe
try:
    model
except NameError:
    raise RuntimeError("Objeto 'model' não encontrado no ambiente. Construa/import o modelo antes de rodar este script.")

# ---------- Utilities ----------
def clean_name(name):
    if name is None:
        return ""
    if CLEAN_MODULE_PREFIX and name.startswith("module."):
        return name[len("module."):]
    return name

def freeze_prefixes(model, prefixes):
    if not prefixes:
        return
    for mname, module in model.named_modules():
        for p in prefixes:
            if mname.startswith(p):
                for param in module.parameters():
                    param.requires_grad = False
    print(f"Froze prefixes: {prefixes}")

# ---------- Optional: freeze requested modules ----------
if PREFIXES_TO_FREEZE:
    freeze_prefixes(model, PREFIXES_TO_FREEZE)

# ---------- Gather named parameters ----------
param_rows = []  # list of tuples (name, shape, numel, trainable_bool)
for name, p in model.named_parameters():
    cname = clean_name(name)
    param_rows.append((cname, tuple(p.shape), int(p.numel()), bool(p.requires_grad)))

# Save per-parameter CSV
with open(OUT_CSV_PER_PARAM, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ParamName", "Shape", "Numel", "Trainable"])
    for name, shape, numel, trainable in param_rows:
        writer.writerow([name, str(shape), numel, int(trainable)])
print(f"Wrote per-parameter CSV: {OUT_CSV_PER_PARAM} (rows: {len(param_rows)})")

# ---------- Group by prefix depth ----------
def group_by_prefix(param_rows, depth=2):
    d = defaultdict(lambda: [0,0])  # key -> [total, trainable]
    for name, shape, numel, trainable in param_rows:
        if name == "":
            key = "(root)"
        else:
            parts = name.split(".")
            # If depth > len(parts) use full name
            key = ".".join(parts[:depth]) if len(parts) >= depth else name
        d[key][0] += numel
        d[key][1] += (numel if trainable else 0)
    # sort by total descending
    ordered = OrderedDict(sorted(d.items(), key=lambda x: x[1][0], reverse=True))
    return ordered

grouped = group_by_prefix(param_rows, depth=DEPTH)

# Optionally trim to top_k
items = list(grouped.items())
if TOP_K is not None:
    items = items[:TOP_K]

# Write grouped CSV
with open(OUT_CSV_BY_MODULE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ModulePrefix", "TotalParams", "TrainableParams"])
    for k, (tot, tr) in items:
        writer.writerow([k, tot, tr])
print(f"Wrote grouped CSV (depth={DEPTH}): {OUT_CSV_BY_MODULE} (rows: {len(items)})")

# ---------- Print summary to notebook ----------
total_params = sum(r[2] for r in param_rows)
trainable_params = sum(r[3] for r in param_rows)
print(f"\nSUMMARY: total parameters: {total_params:,}, trainable: {trainable_params:,}")
print(f"Group-by depth={DEPTH} top entries:")
for i,(k,(tot,tr)) in enumerate(items[:30]):
    print(f"{i+1:02d}. {k:60s} : total {tot:,} | trainable {tr:,}")

# ---------- Optional: helper to produce CSV compatible with latex converter (full rows) ----------
# If you want a CSV with the exact format used by the csv_to_latex_table converter (Block,Layer,...),
# create it like below (one line per grouped module), else skip.

OUT_CSV_LATEX = "params_table_input_from_groups.csv"
with open(OUT_CSV_LATEX, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["Block","Layer","Input Size","Output Size","Params","Activation"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for k,(tot,tr) in items:
        writer.writerow({
            "Block": k,
            "Layer": k,
            "Input Size": "",
            "Output Size": "",
            "Params": f"{tot:,} | {tr:,}",
            "Activation": ""
        })
    # add total row at the end
    writer.writerow({
        "Block": "-",
        "Layer": "Total",
        "Input Size": "",
        "Output Size": "",
        "Params": f"{total_params:,} | {trainable_params:,}",
        "Activation": ""
    })
print(f"Wrote LaTeX-friendly CSV: {OUT_CSV_LATEX}")


Wrote per-parameter CSV: params_by_parameter.csv (rows: 417)
Wrote grouped CSV (depth=4): params_by_module_depth4.csv (rows: 91)

SUMMARY: total parameters: 95,076,178, trainable: 417
Group-by depth=4 top entries:
01. feature_extractor.layers.2.blocks                            : total 56,895,264 | trainable 56,895,264
02. feature_extractor.layers.3.blocks                            : total 25,226,304 | trainable 25,226,304
03. feature_extractor.layers.2.downsample                        : total 2,101,248 | trainable 2,101,248
04. model.DCAMA_blocks.2.linears                                 : total 2,099,200 | trainable 2,099,200
05. model.DCAMA_multhead.0.linears                               : total 2,099,200 | trainable 2,099,200
06. feature_extractor.layers.1.blocks                            : total 1,587,984 | trainable 1,587,984
07. model.mixer1.0.weight                                        : total 1,032,192 | trainable 1,032,192
08. feature_extractor.head.weight              

In [72]:
# ============================
# Script: params_to_csv_with_io.py (Jupyter-ready)
# ============================
# Objetivo: gerar:
#  1) params_by_parameter.csv -> linha por parâmetro (name, shape, numel, trainable)
#  2) params_by_module_depth{depth}.csv -> linhas agregadas por prefixo (depth configurável)
#  3) params_table_input_from_groups.csv -> versão LaTeX-friendly, agora com Input/Output Size
# ============================

import csv
import os
from collections import defaultdict, OrderedDict
import torch

# ====== CONFIGURAÇÃO ======
DEPTH = 2             # profundidade de agrupamento
TOP_K = None          # quantos blocos incluir (None = todos)
PREFIXES_TO_FREEZE = []  # ex: ['backbone', 'u2net']
CLEAN_MODULE_PREFIX = True
OUT_CSV_PER_PARAM = "params_by_parameter.csv"
OUT_CSV_BY_MODULE = f"params_by_module_depth{DEPTH}.csv"
OUT_CSV_LATEX = "params_table_input_from_groups.csv"

# Detecta se há GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move o modelo para GPU/CPU
model = model.to(device)

# Ajusta entradas
H, W = 384, 384  # ou pegue dinamicamente do modelo como comentei antes
query_img   = torch.randn(1, 3, H, W, device=device)
support_img = torch.randn(1, 3, H, W, device=device)
support_mask = torch.randint(0, 2, (1, 1, H, W), device=device).float()

# =========================================

# ---------- Funções auxiliares ----------
def clean_name(name):
    if name is None:
        return ""
    if CLEAN_MODULE_PREFIX and name.startswith("module."):
        return name[len("module."):]
    return name

def freeze_prefixes(model, prefixes):
    if not prefixes:
        return
    for mname, module in model.named_modules():
        for p in prefixes:
            if mname.startswith(p):
                for param in module.parameters():
                    param.requires_grad = False
    print(f"Froze prefixes: {prefixes}")

# ---------- Captura shapes com forward hooks ----------
module_io = {}
def register_hooks(model):
    hooks = []
    for name, module in model.named_modules():
        if len(list(module.children())) == 0:  # só módulos "folha"
            def hook(mod, inp, out, name=name):
                try:
                    in_shape = tuple(inp[0].shape) if isinstance(inp, (list, tuple)) else tuple(inp.shape)
                except:
                    in_shape = "?"
                try:
                    out_shape = tuple(out.shape) if hasattr(out, "shape") else "?"
                except:
                    out_shape = "?"
                module_io[name] = (in_shape, out_shape)
            hooks.append(module.register_forward_hook(hook))
    return hooks

# Verifica se modelo existe
try:
    model
except NameError:
    raise RuntimeError("Objeto 'model' não encontrado. Construa/import o modelo antes.")

if PREFIXES_TO_FREEZE:
    freeze_prefixes(model, PREFIXES_TO_FREEZE)

# Run dummy forward to collect IO shapes
hooks = register_hooks(model)
with torch.no_grad():
    _ = model(query_img, support_img, support_mask)
for h in hooks: h.remove()

# ---------- Param count ----------
param_rows = []
for name, p in model.named_parameters():
    cname = clean_name(name)
    param_rows.append((cname, tuple(p.shape), int(p.numel()), bool(p.requires_grad)))

# Save per-parameter CSV
with open(OUT_CSV_PER_PARAM, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ParamName", "Shape", "Numel", "Trainable"])
    for name, shape, numel, trainable in param_rows:
        writer.writerow([name, str(shape), numel, int(trainable)])
print(f"Wrote per-parameter CSV: {OUT_CSV_PER_PARAM}")

# ---------- Group by prefix ----------
def group_by_prefix(param_rows, depth=2):
    d = defaultdict(lambda: [0,0])  # key -> [total, trainable]
    for name, shape, numel, trainable in param_rows:
        if name == "":
            key = "(root)"
        else:
            parts = name.split(".")
            key = ".".join(parts[:depth]) if len(parts) >= depth else name
        d[key][0] += numel
        d[key][1] += (numel if trainable else 0)
    ordered = OrderedDict(sorted(d.items(), key=lambda x: x[1][0], reverse=True))
    return ordered

grouped = group_by_prefix(param_rows, depth=DEPTH)
items = list(grouped.items())
if TOP_K is not None:
    items = items[:TOP_K]

with open(OUT_CSV_BY_MODULE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["ModulePrefix", "TotalParams", "TrainableParams"])
    for k, (tot, tr) in items:
        writer.writerow([k, tot, tr])
print(f"Wrote grouped CSV: {OUT_CSV_BY_MODULE}")

# ---------- Build LaTeX-friendly CSV ----------
total_params = sum(r[2] for r in param_rows)
trainable_params = sum(r[2] for r in param_rows if r[3])

with open(OUT_CSV_LATEX, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["Block","Layer","Input Size","Output Size","Params","Activation"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for k,(tot,tr) in items:
        in_shape = module_io.get(k, ("", ""))[0]
        out_shape = module_io.get(k, ("", ""))[1]
        writer.writerow({
            "Block": k,
            "Layer": k,
            "Input Size": str(in_shape),
            "Output Size": str(out_shape),
            "Params": f"{tot:,} | {tr:,}",
            "Activation": ""
        })
    writer.writerow({
        "Block": "-",
        "Layer": "Total",
        "Input Size": "",
        "Output Size": "",
        "Params": f"{total_params:,} | {trainable_params:,}",
        "Activation": ""
    })
print(f"Wrote LaTeX-friendly CSV with IO sizes: {OUT_CSV_LATEX}")


ValueError: Input and output must have the same number of spatial dimensions, but got input with spatial dimensions of [1, 384, 384] and output size of torch.Size([48, 48]). Please provide input tensor in (N, C, d1, d2, ...,dK) format and output size in (o1, o2, ...,oK) format.

In [74]:
query_img = query_img.unsqueeze(0) if query_img.dim() == 3 else query_img
support_img = support_img.unsqueeze(0) if support_img.dim() == 3 else support_img
support_mask = support_mask.unsqueeze(0).unsqueeze(0) if support_mask.dim() == 2 else support_mask
support_mask = support_mask.unsqueeze(1) if support_mask.dim() == 3 else support_mask

print("Query:", query_img.shape)
print("Support:", support_img.shape)
print("Mask:", support_mask.shape)

with torch.no_grad():
    _ = model(query_img, support_img, support_mask)

Query: torch.Size([1, 3, 384, 384])
Support: torch.Size([1, 3, 384, 384])
Mask: torch.Size([1, 1, 384, 384])


ValueError: Input and output must have the same number of spatial dimensions, but got input with spatial dimensions of [1, 384, 384] and output size of torch.Size([48, 48]). Please provide input tensor in (N, C, d1, d2, ...,dK) format and output size in (o1, o2, ...,oK) format.